# STEP 14 — 1단계 백본 확장 (effnetv2_s vs convnextv2_base vs swinv2_base)

## 왜 지금 이걸 하나

지금 파이프라인의 가장 큰 약점은 **헛알림 33.3%** 입니다 — 멀쩡한 개 3마리 중
1마리를 병원에 보냅니다. 그리고 이걸 줄이는 길은 이미 한 번 실측으로 확인됐습니다:

> **1단계 AUROC 가 오르면 같은 recall 에서 헛알림이 줄어듭니다.**
> STEP 10 에서 AUROC +0.164 → 헛알림 59.2% → **33.3%**.

그런데 1단계 백본은 **STEP 6 에서 resnet50 → effnetv2_s 로 바꾼 게 전부**입니다.
2단계에서는 그 뒤로 convnextv2_base / swinv2_base 가 effnetv2_s 계열을 이겼는데
(STEP 12·13), **1단계에서는 아무도 안 해봤습니다.** 그 칸이 비어 있습니다.

| | resnet50 | effnetv2_s | convnextv2_base | swinv2_base |
|---|---|---|---|---|
| 1단계 | STEP 6 ✅ | STEP 6 ✅ **현재** | ❓ | ❓ |
| 2단계 | STEP 12 ✅ | STEP 12 ✅ | STEP 12 ✅ **채택** | STEP 13 ✅ |

## 03d 와 뭐가 다른가 — **입력이 다릅니다**

⚠️ 03d 는 `STAGE1_CROP = "full"` 로 돌았습니다. 그 뒤 **STEP 9-A 에서 `f320` 으로
바뀌었고**(val AUROC +0.1205), 지금 서빙도 f320 입니다. 그래서 03d 를 그대로
다시 돌리면 **쓰지도 않는 입력**으로 백본을 고르게 됩니다.

이 노트북은 `f320` 으로 잽니다. 그래서 03d 의 숫자와 **직접 비교하면 안 됩니다.**
기준선은 여기서 같이 돌리는 `effnetv2_s / photometric` 입니다.

## 증강은 왜 하나만 도나

`photometric` 은 STEP 6 에서 이미 1단계에 채택됐습니다 (흐림 하락 −38.5%p,
AUROC 는 그대로). 다시 물을 이유가 없어서 **백본 축만** 돕니다 — 4칸이 3칸이 되어
그만큼 GPU 시간을 아낍니다.

## 판정 기준 (돌리기 **전에** 못 박습니다 — 규칙 2)

* 기준선: `effnetv2_s / photometric / f320`
* 잡음 폭: **AUROC ±0.01** (`experiments.AUROC_NOISE`, 같은 설정 두 실행 실측)
* **AUROC +0.01 이상** 올라야 교체 후보. 그 아래는 전부 "구분 불가"
* 1등과 2등의 차이도 0.01 안이면 **구분 불가** → 둘 중 **싼 쪽**을 고릅니다
* val AUROC 만 보고 고르지 않습니다. **흐림 하락**을 같이 봅니다 —
  화질 지름길을 쓰는 모델이 val 에서 높게 나오는 걸 STEP 5·6 에서 당했습니다

## 판(batch)으로 나눠 돕니다 — 캐글 할당량 때문

세 종을 한 세션에 못 담습니다 (`convnextv2_base` 가 2단계에서 **에폭당 10.6분**
이었습니다 — 12에폭이면 그것만 2시간 초과).

| 판 | 백본 | 왜 |
|---|---|---|
| **A** | `effnetv2_s`(기준선) + `swinv2_base` | 싼 것 먼저. swinv2 는 2단계에서 convnextv2 와 **구분 불가**(0.0026)인데 **1.6배 쌌습니다** |
| B | `effnetv2_s`(기준선) + `convnextv2_base` | 비싼 쪽. 할당량이 생기면 |

⚠️ **모든 판에 기준선(`effnetv2_s`)이 들어갑니다.** 판을 넘어 AUROC **절대값**을
비교하면 안 되기 때문입니다 — 04 에서 같은 설정 `resnet50` 이 판 A **0.4862** /
판 B **0.5024** (+0.0162) 로 달랐습니다. 시드까지 같은데도요.
판 안에서 "기준선 대비 +얼마" 를 재고, 판끼리는 **그 상승폭만** 견줍니다.

판 A 만으로도 답이 나올 수 있습니다: swinv2 가 기준선을 못 넘으면
"1단계 백본 교체로는 헛알림이 안 줄어든다" 는 결론이고, 판 B 는 그때 가서
할지 말지 정하면 됩니다.

## 한계 (미리 적어둡니다)

* **해상도가 섞여 있습니다.** swinv2_base 는 256px 고정이라 384 로 못 맞춥니다
  (04 판 B 와 같은 관행). 그래서 이건 "백본만" 의 차이가 아니라
  **백본 + 그 백본이 사전학습된 해상도** 의 차이입니다
* 서브셋 55% · 12에폭이라 **절대값은 풀 학습과 다릅니다.** 순위만 봅니다
* holdout 은 안 엽니다. 06 이 풀 학습 뒤에 한 번만 엽니다


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.2"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기 + 사전 검증

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import (labels, split, crop, data, models, train, evaluate,
                 stages, experiments, robust)
from src.config import CLASSES_STAGE1

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS      = 12           # 서브셋 스윕용 (03b·03d 와 동일)
SUBSET      = 0.55         # 학습셋만 줄입니다. 검증셋은 그대로
STAGE1_CROP = "f320"       # ★ STEP 9-A 확정. 03d 는 "full" 이었습니다 — 비교 금지

# ★ 판(batch) — 캐글 할당량이 한 번에 세 종을 못 담아서 나눠 돕니다.
#   ⚠️ **판을 넘어 비교하면 안 됩니다.** 같은 설정 resnet50 이 04 판 A/B 에서
#      0.4862 vs 0.5024 (+0.0162) 로 달랐습니다 — 시드까지 같은데도요.
#      그래서 **모든 판에 기준선(effnetv2_s)을 넣습니다.** 판 안에서는
#      "기준선 대비 +얼마" 를 재고, 판끼리는 그 상승폭만 견줍니다.
BATCH = "A"                             # ← 판을 바꿀 때 여기만 고칩니다

BATCHES = {
    # 판 A — 싼 것 먼저. swinv2 는 2단계에서 convnextv2 와 구분 불가였는데 1.6배 쌌습니다
    "A": [("effnetv2_s", 384), ("swinv2_base", 256)],
    # 판 B — 비싼 것. convnextv2_base 는 2단계에서 **에폭당 10.6분** 이었습니다
    "B": [("effnetv2_s", 384), ("convnextv2_base", 384)],
}

# ★ (백본, 해상도) 짝입니다. ViT 계열은 해상도가 고정이라 CNN 과 같은 384 로
#   못 맞춥니다 (04 판 B 와 같은 관행). 그래서 이건 "백본만" 의 비교가 아닙니다.
MODELS = BATCHES[BATCH]
AUGS   = ("photometric",)               # STEP 6 에서 이미 채택 — 축을 닫았습니다
BASE_MODEL, BASE_AUG = "effnetv2_s", "photometric"
IMG_SIZE = 384                          # 추정·기록용 대표값

# 실측 기준 (STEP 10, **풀 데이터 25에폭 · f320**). 이번은 서브셋 12에폭이라
# 더 낮게 나옵니다. 기준선이 여기서 **크게** 벗어나면 데이터·환경이 달라진 것이므로
# 백본 비교를 읽기 전에 원인부터 찾으세요.
BASE_FULL = {"val_auroc": 0.9530, "holdout_auroc": 0.9304,
             "screening_recall": 0.9458, "false_alarm": 0.333}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")


# ── 학습 전에 전부 확인합니다 ────────────────────────────────────
# 03c 에서 배운 것: 크롭 확인을 학습 루프 안에 두면 78분 뒤에 터집니다.
have = crop.available_tags()
print(f"사용 가능한 크롭 태그: {have}")
if STAGE1_CROP not in have:
    raise SystemExit(
        f"❌ 크롭 '{STAGE1_CROP}' 이 없습니다. 붙어 있는 것: {have}\n"
        f"   Kaggle 우측 [Add Input] 에서 **dogskin-f320** 을 붙이세요.\n"
        f"   (이 노트북은 1단계만 돕니다 — dogskin-m25 는 필요 없습니다)")

d = crop.switch_tag(df, STAGE1_CROP)      # 커버리지 95% 미만이면 여기서 멈춥니다
view = stages.to_stage1(d)
split.verify(view, fold=0, strict=True)   # 누수가 있으면 여기서 에러
tr, va = split.get_fold(view, 0)

N_TRAIN = int(len(tr) * SUBSET)
print(f"\n1단계 뷰 {len(view):,}행  ·  train {len(tr):,} → {N_TRAIN:,}({SUBSET:.0%})"
      f"  ·  val {len(va):,}")
print(f"판 {BATCH} — 조건 {len(MODELS)} × {len(AUGS)} = {len(MODELS) * len(AUGS)}개")
print(f"   {', '.join(f'{m}@{s}' for m, s in MODELS)}")
if BASE_MODEL not in [m for m, _ in MODELS]:
    raise SystemExit(f"❌ 판 {BATCH} 에 기준선 {BASE_MODEL} 이 없습니다 — 비교가 불가능합니다.")

---
## 2. 시작 전에 — 몇 시간 걸릴지 먼저 잽니다

합성 텐서로 GPU 속도만 재므로 **백본당 20초** 안쪽입니다. 학습은 아직 시작 안 합니다.

> 이번 프로젝트에서 "몇 시간 걸릴지 모르고 돌렸다가 뒤통수" 를 여러 번 맞았습니다.
> 여기서 총 예상 시간을 보고 **너무 길면 그만두거나 서브셋을 줄이세요.**

⚠️ GPU 속도만 잰 **하한**입니다. 데이터 로딩이 병목이면 실제는 더 걸립니다
(실측: 384px 에서 GPU 112 img/s 상한, 로더 90 img/s).

In [ ]:
est = experiments.estimate_runtime(
    MODELS, img_size=IMG_SIZE, n_train=N_TRAIN, epochs=EPOCHS,
    n_conditions=len(MODELS) * len(AUGS))

# ★ 캐글 할당량이 빠듯하면 **여기서 멈추고 판단하세요.**
#    추정은 GPU 속도만 잰 하한이라 실제는 더 걸립니다 (STEP 6: 추정 2.3h → 실제 3.3h).
QUOTA_H = 4.4        # ← 지금 남은 캐글 GPU 시간 (직접 고치세요)
_need = est["total_hours"] * 1.45 + 0.3      # 로딩 병목(STEP 6 실측 1.45배) + 교란 검사
print(f"\n  남은 할당량 {QUOTA_H:.1f}h  vs  필요 추정 {_need:.1f}h")

if _need > QUOTA_H:
    print("  ❌ 모자랍니다. ⚠️ 도중에 끊기면 그 Commit 은 **출력이 안 남습니다.**")
    print("\n     줄이는 순서 —")
    print("     ① BATCH 를 'A' 로 (effnetv2_s + swinv2_base 둘만)")
    print("        기준선이 판 안에 있으므로 **이것만으로도 비교는 성립합니다.**")
    print("        convnextv2 는 판 B 로 미룹니다 — 없어지는 게 아닙니다.")
    print("     ② SUBSET 0.55 → 0.35   (학습셋만. 검증셋은 그대로)")
    print("     ③ 그래도 안 되면 이번 주는 접고 다음 주 06 에 시간을 쓰세요.")
    raise SystemExit("할당량 부족 — 위 순서대로 줄이고 이 셀을 다시 도세요.")

print("  ✅ 들어갑니다.")
_slow = max(est["rows"], key=lambda r: r["run_min"], default=None)
if _slow:
    print(f"     제일 비싼 것: {_slow['model']} — 1에폭 {_slow['epoch_min']:.1f}분 "
          f"× {EPOCHS} = {_slow['run_min']:.0f}분")

# VRAM 이 빠듯하면 배치가 자동으로 줄어듭니다. 14.6GB(T4) 를 넘으면 OOM 위험.
for _r in est["rows"]:
    if (_r.get("peak_vram_gb") or 0) > 13.0:
        print(f"🚨 {_r['model']} 이 VRAM {_r['peak_vram_gb']}GB 를 씁니다 — OOM 위험.")
        print("   config 의 batch_size 를 직접 낮추거나 grad_accum 을 올리세요.")


---
## 3. 2×2 학습

⚠️ 여기서부터 오래 걸립니다. 위 예상 시간을 보고 진행하세요.
세션이 끊겨도 `train.fit(resume=True)` 가 마지막 에폭부터 이어받습니다.

In [ ]:
# 위에서 이미 검증한 view 를 씁니다 (여기서 크롭·분할로 실패할 일이 없어야 합니다)
# ★ 이어받기: 끝난 조합은 train.fit 이 건너뜁니다. 세션이 죽으면 이 셀만 다시 도세요.
import gc

runs = []
for _m, _size in MODELS:
    for _a in AUGS:
        try:
            runs.append(experiments.train_and_measure(
                view, stage=1, img_size=_size, crop_tag=STAGE1_CROP,
                device=DEV, epochs=EPOCHS, model_name=_m, aug=_a,
                subset_frac=SUBSET,
                # 배율 교란은 1단계에서 이미 통과했습니다 (STEP 4A).
                # 지금 궁금한 건 **화질** 이므로 그쪽만 잽니다 — 시간도 아낍니다.
                measure_robust=False, measure_blur=True, n_robust=2000))
        except torch.cuda.OutOfMemoryError:
            print(f"❌ {_m}: VRAM 부족 — 건너뜁니다 (해상도를 낮춰 다시 시도해 보세요)")
        except Exception as exc:                                    # noqa: BLE001
            print(f"❌ {_m}: {type(exc).__name__}: {exc}")
        finally:
            gc.collect()
            if DEV == "cuda":
                torch.cuda.empty_cache()

# ⚠️ 전부 실패해도 다음 셀로 넘어가면 "결과 0개" 를 몇 시간 뒤에 알게 됩니다.
if not runs:
    raise SystemExit("❌ 성공한 실행이 0개입니다. 위 오류 메시지를 확인하세요.")
if not any(r["model_name"] == BASE_MODEL for r in runs):
    raise SystemExit(f"❌ 기준선({BASE_MODEL})이 실패했습니다 — 비교할 기준이 없습니다.")


---
## 4. 판정

기준은 `experiments.stage1_report()` 에 있습니다 — **실험 전에** 정해뒀고
노트북 셀이 아니라 `src/` 에 있어서 `git pull` 로 갱신됩니다 (규칙 2·3).

In [ ]:
verdict = experiments.stage1_report(runs, base_model=BASE_MODEL, base_aug=BASE_AUG)

import json
W = env.work_root(); (W/"reports").mkdir(parents=True, exist_ok=True)
keep = ("stage", "model_name", "aug", "img_size", "crop_tag", "exp_name",
        "epochs", "batch_size", "minutes", "best_epoch", "n_epochs", "converged",
        "subset_frac", "n_train", "auroc", "threshold", "precision",
        "blur_drop", "blur_worst", "blur_worst_at")
(W/"reports"/f"step14_stage1_backbone_{BATCH}.json").write_text(json.dumps({
    "step": "STEP14_1단계_백본확장",
    "batch": BATCH,                      # ★ 어느 판이었는지 — 판을 넘어 비교 금지
    "img_size_by_model": {m: s for m, s in MODELS},
    "epochs": EPOCHS, "subset_frac": SUBSET,
    "crop": STAGE1_CROP, "baseline_full_run": BASE_FULL,
    "baseline": f"{BASE_MODEL}/{BASE_AUG}",
    "noise": {"auroc": experiments.AUROC_NOISE, "blur_pp": experiments.BLUR_NOISE_PP},
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
    "verdict": verdict,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n저장: {W/'reports'/f'step14_stage1_backbone_{BATCH}.json'}")

print("\n다음 —")
print(f"  다른 판을 돌리려면 4번 셀의 BATCH 를 바꿔 **새 Commit** 으로 돌리세요.")
print("  ⚠️ 판끼리 AUROC **절대값**을 비교하지 마세요. 기준선 대비 **상승폭**만 견줍니다.")
if verdict.get("model_tie"):
    print("  1·2등이 잡음 안입니다 → **싼 쪽**을 고르세요 (분 컬럼 참고).")
print(f"  06 의 1단계 백본을 '{verdict.get('best', {}).get('model', BASE_MODEL)}' 로 두고 풀 학습.")
print("  ⚠️ 여기 숫자는 서브셋입니다. 보고서에는 풀 학습 숫자를 쓰세요.")

# 다음 노트북(풀 학습)이 쓸 수 있게 꾸러미로 내보냅니다
train.export_release(
    exps=[r["exp_name"] for r in runs],
    meta={"실험": f"STEP 14 1단계 백본 확장 · 판 {BATCH}", "크롭": STAGE1_CROP,
          "해상도": str({m: s for m, s in MODELS}),
          "서브셋": f"{SUBSET:.0%}", "에폭": EPOCHS,
          "고른 백본": str(verdict.get("best", "판정 없음"))},
    files={f"reports/step14_stage1_backbone_{BATCH}.json": json.loads(
        (W/"reports"/f"step14_stage1_backbone_{BATCH}.json").read_text(encoding="utf-8"))},
)


---
## 5. 다음

**이 노트북은 후보를 고르는 것까지입니다.** 서브셋 55% · 12에폭이라 절대값은
풀 학습과 다릅니다.

| 결과 | 다음 |
|---|---|
| 어느 축이든 채택됨 | 그 조합으로 **03 을 풀 데이터 재실행** → holdout 을 다시 엽니다 |
| 둘 다 잡음 안 | 두 축 모두 닫고 `f320` 재크롭(구도 불일치 가설)으로 |

⚠️ **holdout 은 아직 안 봤습니다.** 풀 학습 뒤에 한 번만 엽니다.
지금 holdout 을 보고 조합을 고르면 그 숫자는 더 이상 정직하지 않습니다.

⚠️ Kaggle 이면 **[Save Version] → Save & Run All (Commit)** 으로 돌리고,
끝나면 `release` 폴더를 **New Dataset (Private)** 으로 만들어 두세요.
`READ_ME_FIRST.txt` 에 어떤 실험인지 적혀 있습니다.